# Partie 1 -- Nettoyage et Preparation des Donnees

## Prudential Life Insurance Assessment

**Objectif** : Predire le niveau de risque (`Response`, variable ordinale a 8 niveaux) des candidats a une assurance vie a partir de plus de 100 variables decrivant leurs profils.  
**Source** : [Kaggle](https://www.kaggle.com/c/prudential-life-insurance-assessment)

---

### Plan

1. Chargement des donnees
2. Suppression des colonnes avec >95% de valeurs manquantes (avec verification)
3. Imputation des valeurs manquantes
4. Encodage des variables textuelles (Label Encoding)
5. Target Encoding pour les colonnes a haute cardinalite
6. Feature Engineering
7. Sauvegarde

---
## Installation des dependances

In [1]:
!pip install pandas numpy scikit-learn


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Imports et Configuration

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import KFold
import warnings
import os

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)

### Definition des types de variables (selon la documentation Kaggle)

In [3]:
CATEGORICAL_COLS = [
    "Product_Info_1", "Product_Info_2", "Product_Info_3", "Product_Info_5",
    "Product_Info_6", "Product_Info_7",
    "Employment_Info_2", "Employment_Info_3", "Employment_Info_5",
    "InsuredInfo_1", "InsuredInfo_2", "InsuredInfo_3", "InsuredInfo_4",
    "InsuredInfo_5", "InsuredInfo_6", "InsuredInfo_7",
    "Insurance_History_1", "Insurance_History_2", "Insurance_History_3",
    "Insurance_History_4", "Insurance_History_7", "Insurance_History_8",
    "Insurance_History_9",
    "Family_Hist_1",
    "Medical_History_2", "Medical_History_3", "Medical_History_4",
    "Medical_History_5", "Medical_History_6", "Medical_History_7",
    "Medical_History_8", "Medical_History_9", "Medical_History_11",
    "Medical_History_12", "Medical_History_13", "Medical_History_14",
    "Medical_History_16", "Medical_History_17", "Medical_History_18",
    "Medical_History_19", "Medical_History_20", "Medical_History_21",
    "Medical_History_22", "Medical_History_23", "Medical_History_25",
    "Medical_History_26", "Medical_History_27", "Medical_History_28",
    "Medical_History_29", "Medical_History_30", "Medical_History_31",
    "Medical_History_33", "Medical_History_34", "Medical_History_35",
    "Medical_History_36", "Medical_History_37", "Medical_History_38",
    "Medical_History_39", "Medical_History_40", "Medical_History_41",
]

CONTINUOUS_COLS = [
    "Product_Info_4", "Ins_Age", "Ht", "Wt", "BMI",
    "Employment_Info_1", "Employment_Info_4", "Employment_Info_6",
    "Insurance_History_5",
    "Family_Hist_2", "Family_Hist_3", "Family_Hist_4", "Family_Hist_5",
]

DISCRETE_COLS = [
    "Medical_History_1", "Medical_History_10", "Medical_History_15",
    "Medical_History_24", "Medical_History_32",
]

DUMMY_COLS = [f"Medical_Keyword_{i}" for i in range(1, 49)]

TARGET = "Response"

---
## 1. Chargement des donnees

In [10]:
DATA_DIR = "prudential-life-insurance-assessment"
SYNTHETIC_FILE = "train_augmented_ctgan_strategy_b.csv"

TRAIN_PATH = os.path.join(DATA_DIR, "train.csv", "train.csv")
SYNTHETIC_PATH = os.path.join(DATA_DIR, "synthetic", SYNTHETIC_FILE)
TEST_PATH  = os.path.join(DATA_DIR, "test.csv", "test.csv")

train = pd.read_csv(TRAIN_PATH)
synthetic = pd.read_csv(SYNTHETIC_PATH)
test  = pd.read_csv(TEST_PATH)

# À retirer car elles seront recalculées
synthetic.drop(columns=["Medical_Keyword_Count", "BMI_Age", "Wt_Ht_ratio"], inplace=True)

print(f"Train : {train.shape[0]:,} lignes x {train.shape[1]} colonnes")
print(f"Synthetic : {synthetic.shape[0]:,} lignes x {synthetic.shape[1]} colonnes")
print(f"Test  : {test.shape[0]:,} lignes x {test.shape[1]} colonnes")

# Combiner train + test pour des transformations coherentes
train["is_train"] = 1
synthetic["is_train"] = 1
test["is_train"]  = 0

# Flag pour identifier les données synthétiques
train["is_synthetic"] = 0
synthetic["is_synthetic"] = 1
test["is_synthetic"]  = 0


if TARGET not in test.columns:
    test[TARGET] = np.nan

df = pd.concat([train, synthetic, test], axis=0, ignore_index=True)
print(f"Jeu combine : {df.shape[0]:,} lignes x {df.shape[1]} colonnes")

Train : 59,381 lignes x 128 colonnes
Synthetic : 96,531 lignes x 128 colonnes
Test  : 19,765 lignes x 127 colonnes
Jeu combine : 175,677 lignes x 130 colonnes


---
## 2. Suppression des colonnes avec >95% de valeurs manquantes

Avant de supprimer, on verifie si les quelques valeurs presentes dans ces colonnes
ont un lien significatif avec la variable cible `Response`.  
Si dans les 5% restants il y a un signal fort, la colonne est precieuse malgre le vide.

In [12]:
MISSING_THRESHOLD = 0.95

feature_cols = [c for c in df.columns if c not in ["Id", "is_train", "is_synthetic", TARGET]]
missing_pct = (df[feature_cols].isnull().sum() / len(df) * 100).round(2)

candidates_to_drop = missing_pct[missing_pct > MISSING_THRESHOLD * 100].index.tolist()

print(f"Colonnes candidates a la suppression (>{MISSING_THRESHOLD*100:.0f}% manquant) : {len(candidates_to_drop)}")
for c in candidates_to_drop:
    print(f"   - {c} ({missing_pct[c]:.1f}% manquant)")

Colonnes candidates a la suppression (>95% manquant) : 0


In [13]:
# Verification : est-ce que les valeurs non-manquantes sont correlees a Response ?
# On utilise uniquement le train (le test n'a pas de Response)
df_train = df[df["is_train"] == 1].copy()
global_mean_response = df_train[TARGET].mean()

print(f"Moyenne globale de Response : {global_mean_response:.3f}")
print(f"{'─' * 80}")
print(f"{'Colonne':<25} {'% NaN':>8} {'N present':>10} {'Moy Response (present)':>25} {'Moy Response (absent)':>25} {'Ecart':>8}")
print(f"{'─' * 80}")

cols_to_drop = []
cols_to_keep = []

for col in candidates_to_drop:
    present_mask = df_train[col].notna()
    n_present = present_mask.sum()
    
    if n_present < 10:
        # Trop peu de donnees pour juger -> supprimer
        cols_to_drop.append(col)
        print(f"{col:<25} {missing_pct[col]:>7.1f}% {n_present:>10} {'trop peu de donnees':>25} {'':>25} {'DROP':>8}")
        continue
    
    mean_present = df_train.loc[present_mask, TARGET].mean()
    mean_absent  = df_train.loc[~present_mask, TARGET].mean()
    ecart = abs(mean_present - mean_absent)
    
    # Si l'ecart entre present/absent est > 1 point de Response, c'est un signal
    if ecart > 1.0:
        decision = "KEEP"
        cols_to_keep.append(col)
    else:
        decision = "DROP"
        cols_to_drop.append(col)
    
    print(f"{col:<25} {missing_pct[col]:>7.1f}% {n_present:>10} {mean_present:>25.3f} {mean_absent:>25.3f} {decision:>8}")

print(f"\n{'─' * 80}")
print(f"Resultat : {len(cols_to_drop)} colonnes a supprimer, {len(cols_to_keep)} colonnes a garder")
if cols_to_keep:
    print(f"Colonnes gardees malgre >95% NaN (signal fort) : {cols_to_keep}")

Moyenne globale de Response : 4.500
────────────────────────────────────────────────────────────────────────────────
Colonne                      % NaN  N present    Moy Response (present)     Moy Response (absent)    Ecart
────────────────────────────────────────────────────────────────────────────────

────────────────────────────────────────────────────────────────────────────────
Resultat : 0 colonnes a supprimer, 0 colonnes a garder


In [14]:
# Suppression uniquement des colonnes sans signal
df.drop(columns=cols_to_drop, inplace=True)

CATEGORICAL_COLS = [c for c in CATEGORICAL_COLS if c not in cols_to_drop]
CONTINUOUS_COLS  = [c for c in CONTINUOUS_COLS if c not in cols_to_drop]
DISCRETE_COLS    = [c for c in DISCRETE_COLS if c not in cols_to_drop]
DUMMY_COLS       = [c for c in DUMMY_COLS if c not in cols_to_drop]

print(f"{len(cols_to_drop)} colonnes supprimees.")
print(f"Dimensions apres suppression : {df.shape}")

0 colonnes supprimees.
Dimensions apres suppression : (175677, 130)


---
## 3. Imputation des valeurs manquantes

| Type | Strategie | Justification |
|:-----|:----------|:--------------|
| Continue (BMI, Age...) | Mediane | Robuste aux valeurs extremes |
| Discrete (Medical History) | -1 | L'absence d'info est un signal en soi |
| Categorielle (codes medicaux) | -1 | Le modele apprend que "manquant" = categorie a part |

In [15]:
# Variables continues -> Mediane
continuous_present = [c for c in CONTINUOUS_COLS if c in df.columns]
for col in continuous_present:
    if df[col].isnull().any():
        df[col].fillna(df[col].median(), inplace=True)

# Variables discretes -> -1
discrete_present = [c for c in DISCRETE_COLS if c in df.columns]
for col in discrete_present:
    if df[col].isnull().any():
        df[col].fillna(-1, inplace=True)

# Variables categorielles -> -1
categorical_present = [c for c in CATEGORICAL_COLS if c in df.columns]
for col in categorical_present:
    if df[col].isnull().any():
        df[col].fillna(-1, inplace=True)

# Imputation de securite pour tout ce qui reste
remaining_features = [c for c in df.columns if c not in ["Id", "is_train", "is_synthetic", TARGET]]
for col in remaining_features:
    if df[col].isnull().any():
        if df[col].dtype == "object":
            df[col].fillna("Missing", inplace=True)
        else:
            df[col].fillna(df[col].median(), inplace=True)

nan_count = df[remaining_features].isnull().sum().sum()
print(f"Valeurs manquantes restantes : {nan_count}")

Valeurs manquantes restantes : 0


---
## 4. Encodage des variables textuelles (Label Encoding)

La colonne `Product_Info_2` contient des codes textuels ("A1", "D3"...). On les transforme en entiers.

In [16]:
text_cols = df[remaining_features].select_dtypes(include=["object"]).columns.tolist()

for col in text_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    print(f"{col} : {len(le.classes_)} categories -> [{df[col].min()}, {df[col].max()}]")

if not text_cols:
    print("Aucune colonne textuelle a encoder.")

Product_Info_2 : 38 categories -> [0, 37]


---
## 5. Target Encoding pour les colonnes a haute cardinalite

On remplace chaque categorie par la moyenne de `Response` pour cette categorie.  
K-Fold (5 folds) pour eviter le data leakage.

In [17]:
HIGH_CARDINALITY_THRESHOLD = 20


def target_encode_kfold(df, col, target, n_splits=5):
    """Target Encoding avec K-Fold pour eviter le data leakage."""
    train_mask = df["is_train"] == 1
    test_mask  = df["is_train"] == 0

    encoded_col = f"{col}_target_enc"
    df[encoded_col] = np.nan

    train_idx = df[train_mask].index
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

    for train_fold_idx, val_fold_idx in kf.split(train_idx):
        actual_train_idx = train_idx[train_fold_idx]
        actual_val_idx   = train_idx[val_fold_idx]
        means = df.loc[actual_train_idx].groupby(col)[target].mean()
        df.loc[actual_val_idx, encoded_col] = df.loc[actual_val_idx, col].map(means)

    global_means = df[train_mask].groupby(col)[target].mean()
    global_mean  = df.loc[train_mask, target].mean()
    df.loc[test_mask, encoded_col] = df.loc[test_mask, col].map(global_means)
    df[encoded_col].fillna(global_mean, inplace=True)

    return encoded_col

In [18]:
all_cat_cols = [c for c in categorical_present + discrete_present if c in df.columns]
high_card_cols = [c for c in all_cat_cols if df[c].nunique() > HIGH_CARDINALITY_THRESHOLD]

print(f"Colonnes a haute cardinalite (>{HIGH_CARDINALITY_THRESHOLD} categories) : {len(high_card_cols)}")

for col in high_card_cols:
    n_unique = df[col].nunique()
    new_col = target_encode_kfold(df, col, TARGET)
    print(f"   {col} ({n_unique} categories) -> {new_col}")

if high_card_cols:
    df.drop(columns=high_card_cols, inplace=True)
    print(f"Colonnes originales supprimees.")

Colonnes a haute cardinalite (>20 categories) : 9
   Product_Info_2 (38 categories) -> Product_Info_2_target_enc
   Product_Info_3 (38 categories) -> Product_Info_3_target_enc
   Employment_Info_2 (38 categories) -> Employment_Info_2_target_enc
   Medical_History_2 (648 categories) -> Medical_History_2_target_enc
   Medical_History_1 (200 categories) -> Medical_History_1_target_enc
   Medical_History_10 (222 categories) -> Medical_History_10_target_enc
   Medical_History_15 (242 categories) -> Medical_History_15_target_enc
   Medical_History_24 (242 categories) -> Medical_History_24_target_enc
   Medical_History_32 (107 categories) -> Medical_History_32_target_enc
Colonnes originales supprimees.


---
## 6. Feature Engineering

In [19]:
med_kw_cols = [c for c in DUMMY_COLS if c in df.columns]
if med_kw_cols:
    df["Medical_Keyword_Count"] = df[med_kw_cols].sum(axis=1)

if "BMI" in df.columns and "Ins_Age" in df.columns:
    df["BMI_Age"] = df["BMI"] * df["Ins_Age"]

if "Wt" in df.columns and "Ht" in df.columns:
    df["Wt_Ht_ratio"] = df["Wt"] / (df["Ht"] + 1e-8)

print("Features ajoutees : Medical_Keyword_Count, BMI_Age, Wt_Ht_ratio")

Features ajoutees : Medical_Keyword_Count, BMI_Age, Wt_Ht_ratio


---
## 7. Sauvegarde

In [21]:
train_clean = df[(df["is_train"] == 1) & (df["is_synthetic"] == 0)].drop(columns=["is_train", "is_synthetic"])
synthetic_clean = df[df["is_synthetic"] == 1].drop(columns=["is_train", "is_synthetic"])
test_clean  = df[df["is_train"] == 0].drop(columns=["is_train", "is_synthetic", TARGET])

feature_cols_final = [c for c in train_clean.columns if c not in ["Id", TARGET]]
train_nan = train_clean[feature_cols_final].isnull().sum().sum()
synthetic_nan = synthetic_clean[feature_cols_final].isnull().sum().sum()
test_nan  = test_clean.drop(columns=["Id"]).isnull().sum().sum()

print(f"Train : {train_clean.shape[0]:,} x {train_clean.shape[1]} | NaN : {train_nan}")
print(f"Synthetic : {synthetic_clean.shape[0]:,} x {synthetic_clean.shape[1]} | NaN : {synthetic_nan}")
print(f"Test  : {test_clean.shape[0]:,} x {test_clean.shape[1]} | NaN : {test_nan}")
print(f"Features : {len(feature_cols_final)}")

Train : 59,381 x 131 | NaN : 0
Synthetic : 96,531 x 131 | NaN : 0
Test  : 19,765 x 130 | NaN : 0
Features : 129


In [22]:
OUTPUT_DIR = os.path.join(DATA_DIR, "synthetic", "cleaned", SYNTHETIC_FILE.replace(".csv", ""))

train_output = os.path.join(OUTPUT_DIR, "train_clean.csv")
test_output  = os.path.join(OUTPUT_DIR, "test_clean.csv")
synthetic_output = os.path.join(OUTPUT_DIR, SYNTHETIC_FILE.replace(".csv", "_clean.csv"))

os.makedirs(OUTPUT_DIR, exist_ok=True)

train_clean.to_csv(train_output, index=False)
test_clean.to_csv(test_output, index=False)
synthetic_clean.to_csv(synthetic_output, index=False)

print(f"Sauvegardes : {train_output}, {test_output}, {synthetic_output}")

Sauvegardes : prudential-life-insurance-assessment\synthetic\cleaned\train_augmented_ctgan_strategy_b\train_clean.csv, prudential-life-insurance-assessment\synthetic\cleaned\train_augmented_ctgan_strategy_b\test_clean.csv, prudential-life-insurance-assessment\synthetic\cleaned\train_augmented_ctgan_strategy_b\train_augmented_ctgan_strategy_b_clean.csv


In [23]:
synthetic_clean.head()

,Id,Product_Info_1,Product_Info_4,Product_Info_5,Product_Info_6,Product_Info_7,Ins_Age,Ht,Wt,BMI,Employment_Info_1,Employment_Info_3,Employment_Info_4,Employment_Info_5,Employment_Info_6,InsuredInfo_1,InsuredInfo_2,InsuredInfo_3,InsuredInfo_4,InsuredInfo_5,InsuredInfo_6,InsuredInfo_7,Insurance_History_1,Insurance_History_2,Insurance_History_3,...,Medical_Keyword_37,Medical_Keyword_38,Medical_Keyword_39,Medical_Keyword_40,Medical_Keyword_41,Medical_Keyword_42,Medical_Keyword_43,Medical_Keyword_44,Medical_Keyword_45,Medical_Keyword_46,Medical_Keyword_47,Medical_Keyword_48,Response,Product_Info_2_target_enc,Product_Info_3_target_enc,Employment_Info_2_target_enc,Medical_History_2_target_enc,Medical_History_1_target_enc,Medical_History_10_target_enc,Medical_History_15_target_enc,Medical_History_24_target_enc,Medical_History_32_target_enc,Medical_Keyword_Count,BMI_Age,Wt_Ht_ratio
59381,79147,1,0.074934,2,3,1,0.221048,0.772566,0.239752,0.672064,0.073737,3,0.042682,2,0.232070,1,2,3,3,1,2,1,2,1,3,...,0,0,0,0,0,0,0,0,0,0,0,0,1.0,3.525746,4.400933,4.130328,3.623550,4.831825,4.498680,4.783406,4.765086,4.499081,2,0.148558,0.310332
59382,79148,1,1.000000,2,3,1,0.199253,0.594426,0.147176,0.471412,0.000000,3,0.030659,2,0.218292,1,2,3,3,1,2,1,2,1,3,...,0,0,0,0,0,0,0,0,0,0,0,0,1.0,3.597554,4.403320,4.732859,2.569767,4.844550,4.503579,4.790056,4.769388,4.505283,1,0.093930,0.247593
59383,79149,1,0.232547,2,3,1,0.299794,0.863257,0.185172,0.553593,0.036625,1,0.040834,2,0.223859,2,2,3,3,1,1,1,2,1,3,...,0,0,0,0,0,0,0,0,0,0,0,0,1.0,3.413406,4.410869,4.731035,3.341270,4.411176,4.509101,4.791375,3.044928,4.509506,2,0.165964,0.214504
59384,79150,1,0.435827,2,3,1,0.187345,0.839990,0.064854,0.620008,0.112321,1,0.003716,2,0.730379,1,2,3,3,1,2,1,2,3,3,...,0,0,0,0,0,0,0,0,0,0,1,0,1.0,3.416547,4.414559,4.745600,3.659420,4.420503,4.518819,3.732018,4.780293,4.517007,3,0.116156,0.077208
59385,79151,1,0.096491,2,1,1,0.489188,0.477120,0.178829,1.000000,0.022203,1,0.004193,2,0.044977,1,2,8,3,1,1,1,2,1,3,...,0,0,0,0,0,0,0,0,0,0,1,0,1.0,3.419905,4.402467,4.113954,4.133333,4.427011,4.503724,4.783527,4.766347,4.503098,2,0.489188,0.374809


---
## Conclusion

| Operation | Detail |
|:----------|:-------|
| Colonnes poubelles | Verifiees puis supprimees (sauf si signal fort avec Response) |
| Imputation | Mediane (continu), -1 (categoriel/discret) |
| Encodage | Label Encoding sur Product_Info_2 |
| Haute cardinalite | Target Encoding K-Fold |
| Feature Engineering | Medical_Keyword_Count, BMI_Age, Wt_Ht_ratio |

**Prochaine etape** : Partie 2 -- Entrainement des modeles XGBoost et CatBoost.